# Knowledge crew: an auto-grounded knowledge base in a dozen lines

The fastest way to put your data behind a CrewAI crew: hand it an
[Infino](https://pypi.org/project/infino/) knowledge store and the agents ground
every answer in it **automatically** — no tools to wire, no retrieval calls to
write. With [`crewai-infino`](https://pypi.org/project/crewai-infino/),
`InfinoKnowledgeStorage` embeds and stores your chunks in **one** table and
serves them back by vector similarity, with no second vector database to keep in
sync.

Building the store is key-free; the crew is tool-calling, so it needs an LLM key —
gated below. See the README.

In [1]:
import shutil
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # examples/ root, where _shared/ lives

import infino
from crewai import Agent, Crew, Process, Task
from crewai.knowledge.knowledge import Knowledge
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
from crewai_infino import InfinoKnowledgeStorage

from _shared.crew import crew_llm
from _shared.embedding import DIM, embed, embed_query
from _shared.sql import query

DB_DIR = "./knowledge_data"
shutil.rmtree(DB_DIR, ignore_errors=True)  # start fresh each run

db = infino.connect(DB_DIR)
llm = crew_llm()
print("LLM:", "on" if llm else "off (build + search still run; crew is skipped)")

LLM: on


## Build the knowledge base

`InfinoKnowledgeStorage` is a CrewAI knowledge backend over one Infino table.
Wrap your facts in a `StringKnowledgeSource`, hand both to `Knowledge`, and
`add_sources()` embeds and stores them — that's the whole setup. (Swap in
CrewAI's PDF / file sources to ground a crew in your own documents the same way.)

In [2]:
storage = InfinoKnowledgeStorage(
    connection=db, embed_documents=embed, embed_query=embed_query, dim=DIM,
)
FACTS = [
    "The NovaSound Z9 earbuds ship with a 30-hour battery and an IPX5 sweatproof rating.",
    "Error E-507 on the NovaSound Z9 means a stale Bluetooth cache: forget the device "
    "and re-pair to clear it.",
    "The HydroPeak 32oz bottle keeps drinks cold for 24 hours; the lid is top-rack "
    "dishwasher safe.",
]
knowledge = Knowledge(
    collection_name="kb",
    sources=[StringKnowledgeSource(content=f) for f in FACTS],
    storage=storage,
)
knowledge.add_sources()  # embed + store every fact in one Infino table
print("stored", query(db, "SELECT COUNT(*) AS n FROM knowledge")["n"][0], "facts")

stored 3 facts


## Retrieval works before any LLM

The store is queryable on its own — vector similarity over the same table,
key-free. Scores are higher-is-better (cosine similarity).

In [3]:
for hit in storage.search(["how long does the Z9 battery last?"], score_threshold=0.1):
    print(f"{hit['score']:.2f}  {hit['content'][:70]}")

0.49  The NovaSound Z9 earbuds ship with a 30-hour battery and an IPX5 sweat
0.28  Error E-507 on the NovaSound Z9 means a stale Bluetooth cache: forget 
0.11  The HydroPeak 32oz bottle keeps drinks cold for 24 hours; the lid is t


## The crew auto-grounds

Give the `Crew` the knowledge and a plain agent — **no tools, no retrieval code**.
The agent answers from the store automatically; the facts it cites appear in no
prompt.

In [4]:
def ask(question: str) -> None:
    if llm is None:
        print(f"Q: {question}\n(needs an LLM key — skipped)")
        return
    agent = Agent(
        role="Product Support Specialist",
        goal="Answer customer questions accurately from the knowledge base.",
        backstory="You help customers with product details.",
        llm=llm, verbose=False,
    )
    task = Task(
        description=question, expected_output="A short, factual answer.", agent=agent
    )
    crew = Crew(agents=[agent], tasks=[task], knowledge=knowledge,
                process=Process.sequential, verbose=False)
    # Notebooks run inside an event loop; run the sync crew in a worker thread.
    with ThreadPoolExecutor(max_workers=1) as pool:
        answer = pool.submit(crew.kickoff).result()
    print(f"Q: {question}\nA: {answer}")


ask("What battery life does the NovaSound Z9 have, and what does error E-507 mean?")

Q: What battery life does the NovaSound Z9 have, and what does error E-507 mean?
A: The NovaSound Z9 has up to 30 hours of battery life. Error E-507 means there’s a stale Bluetooth cache—forget the device and re-pair it to clear the issue.


## Takeaway

A dozen lines gave the crew a knowledge base it grounds every answer in — one
Infino table, embedded and stored automatically, with no tools to wire and no
second vector database to keep in sync.